# sklearn model with tensorflow keras tuner

In [1]:
from helper_func import *
import helper_func as hf
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from spellchecker import SpellChecker
from textblob import TextBlob
from multiprocessing import Pool
from tqdm import tqdm
import numpy as np
import pandas as pd
# Preprocessing
from nltk.tokenize import word_tokenize, sent_tokenize
import operator
from spellchecker import SpellChecker
from tqdm import tqdm  # Import tqdm
import re
import inflect
from wordsegment import load, segment
from nltk.corpus import words
word_list = set(words.words())

2024-04-13 15:59:04.697153: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-04-13 15:59:04.727826: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-13 15:59:05.214856: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
train = pd.read_csv('data/train.csv')

In [3]:
# time the run time

import time

start = time.time()

# Clean the train and test text data

train = hf.clean_text(train, col_name = 'full_text')


end = time.time()

print(f"Time taken to clean text: {end - start} seconds")

Starting text cleaning process. 

Time taken to clean text: 32.43930697441101 seconds


In [4]:
# Apply the parallelization

train = parallelize_dataframe(train, apply_segmentation)

train.head()


  0%|          | 0/20 [01:01<?, ?it/s]
/home/laptop/github/kaggle/scoring/helper_func.py:295: DeprecationWarning: invalid escape sequence '\w'
  text = re.sub("@\w+", '', text)
/home/laptop/github/kaggle/scoring/helper_func.py:296: DeprecationWarning: invalid escape sequence '\d'
  text = re.sub("'\d+", '', text)
/home/laptop/github/kaggle/scoring/helper_func.py:297: DeprecationWarning: invalid escape sequence '\d'
  text = re.sub("\d+", '', text)
/home/laptop/github/kaggle/scoring/helper_func.py:298: DeprecationWarning: invalid escape sequence '\w'
  text = re.sub("http\w+", '', text)


KeyboardInterrupt: 

In [5]:
text_col = 'clean_text'   # 'segmented_text'

In [6]:
glove_path = '/home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt'
paragran_path = '/home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt'
fastetxt_path = '/home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec'


# Rebuild and check vocab after cleaning contractions

train, glove, paragram, fastetxt = embedding_checks(train, glove_path, paragran_path, fastetxt_path, col_name= text_col)

Loading embeddings from /home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vecLoading embeddings from /home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txtLoading embeddings from /home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt




Reading Embedding File: 999995it [02:33, 6505.50it/s] 
Reading Embedding File: 991089it [02:33, 6775.43it/s]

Loaded embeddings from /home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec


Reading Embedding File: 1703756it [04:24, 6447.27it/s] 


Loaded embeddings from /home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt


...

Coverage checked.
Checking coverage.


Checking Words: 100%|██████████| 65918/65918 [00:00<00:00, 380785.19it/s]


Coverage checked.
Checking coverage.


Checking Words: 100%|██████████| 65918/65918 [00:00<00:00, 841628.86it/s]


Coverage checked.
Processed dataset.


In [7]:
misspellings = []

oov = glove + paragram + fastetxt

for word, _ in oov:

    misspellings.append(word)

    misspellings = list(set(misspellings))

print(f"Number of misspelled words: {len(misspellings)}")

# print(f"Misspelled words: {misspellings}")

Number of misspelled words: 36335


In [8]:
# Example usage
corrected_words, uncorrected_words = main(misspellings)

  0%|          | 0/21 [00:04<?, ?it/s]

In [9]:
# Assuming df is your DataFrame and 'clean_text' is the column you want to correct

correction_dict = dict(corrected_words)

train['corrected_text'] = train[text_col].apply(lambda x: apply_corrections_to_text(x, correction_dict))


In [10]:
print(f"Corrected words: {len(corrected_words)}")
print(f"Uncorrected words: {len(uncorrected_words)}")

Corrected words: 31994
Uncorrected words: 4341


In [11]:
train_essays, validation_essays = custom_train_validation_split(train, test_size=0.25, random_state=42)

In [12]:
train_essays, _ = preprocess_data(train_essays)

Preprocessing Data.......
Generating Text Based Features.......


In [13]:
validation_essays, _ = preprocess_data(validation_essays) #, tfidf_vectorizer=tfidf_vectorizer)

Preprocessing Data.......
Generating Text Based Features.......


In [14]:
train_essays.columns

Index(['essay_id', 'full_text', 'score', 'lowered', 'clean_text',
       'corrected_text', 'word_count', 'sentence_count', 'len_text',
       'stop_count', 'noun_count', 'ne_count', 'avg_word_len', 'lex_div',
       'polarity', 'subjectivity', 'flesch_score'],
      dtype='object')

In [15]:
drop_cols = ['full_text','lowered', 'clean_text',
       'corrected_text']

training = train_essays.copy()
validation = validation_essays.copy()

training.drop(columns=drop_cols, inplace= True)
validation.drop(columns=drop_cols, inplace= True)



In [26]:
feature_cols = []

for col in training.columns:
    if (col != 'essay_id') and (col != 'score'):
        feature_cols.append(col) 


# Select relevant columns (replace with actual column names)

training_labels = training['score']

validation_labels = validation['score']

train_features = training[feature_cols].values

val_features = validation[feature_cols].values

# test_features = test[feature_cols].values


# Standardize the features if needed

scaler = StandardScaler()


train_features = scaler.fit_transform(train_features)

val_features = scaler.transform(val_features)


target_scaler = StandardScaler()

# training_labels = target_scaler.fit_transform(training_labels.values.reshape(-1, 1)).flatten()

# validation_labels = target_scaler.transform(validation_labels.values.reshape(-1, 1)).flatten()




# Optionally, convert back to DataFrame

train_labels = pd.DataFrame(training_labels, columns=['score'],
                             index=training_labels.index)

val_labels = pd.DataFrame(validation_labels, columns=['score'], 
                          index=validation_labels.index)

import pickle


with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
    


In [23]:
scaler_path = 'scaler.pkl'

# Check if the file has been written correctly and is not empty
import os

if os.path.getsize(scaler_path) > 0:
    print(f"Scaler saved successfully in {scaler_path}.")
else:
    print(f"Failed to save scaler to {scaler_path}. File is empty.")

Scaler saved successfully in scaler.pkl.


In [24]:
import tensorflow as tf

train_class = tf.data.Dataset.from_tensor_slices((train_features, 
                                                training_labels)).shuffle(len(train_features)).batch(32)

val_class = tf.data.Dataset.from_tensor_slices((val_features, 
                                              validation_labels)).batch(32)


In [35]:
import keras_tuner
from sklearn import ensemble
from sklearn import datasets
from sklearn import linear_model
from sklearn import metrics
from sklearn import model_selection

def build_model(hp):
  model_type = hp.Choice('model_type', ['random_forest', 'ridge'])
  if model_type == 'random_forest':
    model = ensemble.RandomForestClassifier(
        n_estimators=hp.Int('n_estimators', 10, 50, step=10),
        max_depth=hp.Int('max_depth', 3, 10))
  else:
    model = linear_model.RidgeClassifier(
        alpha=hp.Float('alpha', 1e-3, 1, sampling='log'))
  return model

tuner = keras_tuner.tuners.SklearnTuner(
    oracle=keras_tuner.oracles.BayesianOptimizationOracle(
        objective=keras_tuner.Objective('score', 'max'),
        max_trials=100),
    hypermodel=build_model,
    scoring=metrics.make_scorer(metrics.accuracy_score),
    cv=model_selection.StratifiedKFold(10),
    directory='.',
    project_name='my_project')


tuner.search(train_features, train_labels)

best_model = tuner.get_best_models(num_models=1)[0]

Trial 100 Complete [00h 00m 03s]
score: 0.5726502311248074

Best score So Far: 0.5791217257318952
Total elapsed time: 00h 10m 36s


In [36]:
best_model.fit(train_features, train_labels)

RandomForestClassifier(max_depth=8, n_estimators=30)

In [37]:
predictions = best_model.predict(val_features)

# predictions = target_scaler.inverse_transform(predictions.reshape(-1, 1)).flatten()


In [38]:
# confusion matrix

from sklearn.metrics import confusion_matrix

confusion_matrix(val_labels, predictions)

# classification report

from sklearn.metrics import classification_report

print(classification_report(validation_labels, predictions))


              precision    recall  f1-score   support

           1       0.59      0.07      0.12       326
           2       0.65      0.59      0.62      1206
           3       0.57      0.67      0.62      1574
           4       0.54      0.66      0.59       938
           5       0.42      0.33      0.37       239
           6       0.38      0.07      0.12        44

    accuracy                           0.58      4327
   macro avg       0.53      0.40      0.41      4327
weighted avg       0.58      0.58      0.56      4327



In [39]:
# cohens kappa

from sklearn.metrics import cohen_kappa_score

cohen = cohen_kappa_score(val_labels, predictions)

# quadratic weighted kappa

from sklearn.metrics import cohen_kappa_score

quadratic = cohen_kappa_score(val_labels, predictions, weights='quadratic')

print(f"Cohen's Kappa: {cohen}")
print(f"Quadratic Weighted Kappa: {quadratic}")

Cohen's Kappa: 0.4062394116916933
Quadratic Weighted Kappa: 0.6826920104535736


In [41]:
from joblib import dump, load

dump(best_model, 'random_forest.joblib') 


['random_forest.joblib']

In [ ]:
# forest_model = load('random_forest.joblib') 